In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "77262576c836ea69a5f1c2d6be5fadbccf17e3e4"
assert len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH", "Pin the independently reviewed pushed diagnostic implementation commit"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
DIAGNOSTIC_VERSION = "v1_all_class_separability"
KNN_K_VALUES = (1, 5, 10)
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
CLASSIFIER_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier"
V4_FAILURE_RECORD = CLASSIFIER_ROOT / "v4_focal_inverse_frequency" / "latest_validation_failure.json"
EMBEDDING_RECORD = CLASSIFIER_ROOT / "embedding_diagnostics" / "v1_frozen_coca_df_embeddings" / "latest_diagnostic_record.json"
MIXTURE_RECORD = CLASSIFIER_ROOT / "mixture_diagnostics" / "v1_synthetic_mixture_dose_response" / "latest_diagnostic_record.json"
DIAGNOSTIC_ROOT = CLASSIFIER_ROOT / "representation_diagnostics" / DIAGNOSTIC_VERSION
LATEST_RECORD = DIAGNOSTIC_ROOT / "latest_diagnostic_record.json"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".coca_shared_root.json"
EXPECTED_TRAIN_ROWS = 6995
EXPECTED_VALIDATION_ROWS = 1510
EXPECTED_TRAIN_CLASS_COUNTS = {"akiec": 226, "bcc": 348, "bkl": 778, "df": 85, "mel": 782, "nv": 4684, "vasc": 92}
EXPECTED_VALIDATION_CLASS_COUNTS = {"akiec": 45, "bcc": 77, "bkl": 166, "df": 14, "mel": 169, "nv": 1016, "vasc": 23}

# Frozen CoCa all-class embedding separability diagnostic

Descriptive only: uses train and validation splits, never the test split, and authorizes no formal run.


## Phase 0: Pinned code and environment


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys
subprocess.run(["nvidia-smi"], check=True)
from google.colab import userdata
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access is required"
CODE_DIR = Path("/content/ddpm-coca-all-class-separability-code")
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status
assert "@" not in remote and "x-access-token" not in remote
os.environ["HF_HOME"] = "/content/hf-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0", "scikit-learn==1.8.0", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import torch
from ddpm_derm import coca_run
assert torch.cuda.is_available()
assert version("open_clip_torch") == "3.3.0" and version("scikit-learn") == "1.8.0"

## Phase 1: Shared Drive, identity, and prior evidence


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
assert SHARED_ROOT_SENTINEL.is_file(), "existing shared-root sentinel is required"
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
resolved_root = coca_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = coca_run.probe_shared_drive(resolved_root)
sentinel = json.loads(SHARED_ROOT_SENTINEL.read_text(encoding="utf-8"))
assert sentinel["resolved_path"] == str(resolved_root)
shared_root_uuid = sentinel["shared_root_uuid"]
assert V4_FAILURE_RECORD.is_file() and EMBEDDING_RECORD.is_file() and MIXTURE_RECORD.is_file()
guard_paths = (V4_FAILURE_RECORD, EMBEDDING_RECORD, MIXTURE_RECORD)
guard_before = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
v4_failure = json.loads(V4_FAILURE_RECORD.read_text(encoding="utf-8"))
embedding = json.loads(EMBEDDING_RECORD.read_text(encoding="utf-8"))
mixture = json.loads(MIXTURE_RECORD.read_text(encoding="utf-8"))
assert v4_failure["validation_status"] == "VALIDATION FAILED" and v4_failure["formal_training_started"] is False
assert embedding["diagnostic_status"] == "COMPLETED" and embedding["test_data_accessed"] is False
assert embedding["analysis"]["group_counts"] == {"real_train_df": 85, "synthetic_df": 500, "validation_df": 14}
assert mixture["diagnostic_status"] == "COMPLETED"
assert mixture["formal_training_started"] is False and mixture["test_data_accessed"] is False
assert mixture.get("condition_selected", False) is False
assert mixture["interpretation_scope"] == "descriptive_not_candidate_selection"
print("prior evidence verified and guarded:", {str(path): guard_before[str(path)][0][:12] for path in guard_paths})

## Phase 2: Copy train/validation data locally


In [ ]:
import time
import pandas as pd
from datetime import datetime, timezone
LOCAL_DATA_DIR = Path("/content/ham10000-train-val-only")
assert not LOCAL_DATA_DIR.exists()
assert str(LOCAL_DATA_DIR).startswith("/content/") and "drive" not in LOCAL_DATA_DIR.parts
assert SHARED_RUN_ROOT not in LOCAL_DATA_DIR.parents and SHARED_PROJECT_DIR not in LOCAL_DATA_DIR.parents
assert not LATEST_RECORD.exists(), f"completed diagnostic already exists; do not overwrite: {LATEST_RECORD}"
if DIAGNOSTIC_ROOT.exists():
    prior_attempts = [path for path in DIAGNOSTIC_ROOT.iterdir() if path.is_dir()]
    assert not prior_attempts, f"incomplete diagnostic attempt requires manual review: {prior_attempts}"
def copy_group(group, relatives, source_base, target_base, log_every=500):
    total = len(relatives)
    print(f"START {group} copy: total={total}", flush=True)
    start = time.perf_counter()
    for index, relative in enumerate(relatives, start=1):
        source = source_base / relative
        target = target_base / relative
        try:
            if not source.is_file():
                raise FileNotFoundError(f"source image not found: {source}")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target)
        except Exception as exc:
            raise RuntimeError(f"{group} copy failed at {index}/{total} for relative path {relative!r}: {exc}") from exc
        if index % log_every == 0 or index == total:
            elapsed = time.perf_counter() - start
            rate = index / elapsed if elapsed > 0 else float("inf")
            eta = (total - index) / rate if rate > 0 else float("inf")
            print(f"[{group}] {index}/{total} last={relative} elapsed={elapsed:.1f}s rate={rate:.1f} copies/s eta={eta:.1f}s", flush=True)
    elapsed = time.perf_counter() - start
    rate = total / elapsed if elapsed > 0 else float("inf")
    print(f"DONE {group} copy: total={total} elapsed={elapsed:.1f}s avg={rate:.1f} copies/s", flush=True)
(LOCAL_DATA_DIR / "manifests").mkdir(parents=True)
class_mapping_source = SHARED_PROJECT_DIR / "data" / "manifests" / "class_to_idx.json"
assert class_mapping_source.is_file()
shutil.copy2(class_mapping_source, LOCAL_DATA_DIR / "manifests" / "class_to_idx.json")
split_frames = {}
for split, expected_rows, expected_counts in (("train", EXPECTED_TRAIN_ROWS, EXPECTED_TRAIN_CLASS_COUNTS), ("val", EXPECTED_VALIDATION_ROWS, EXPECTED_VALIDATION_CLASS_COUNTS)):
    source_manifest = SHARED_PROJECT_DIR / "data" / "manifests" / f"{split}.csv"
    frame = pd.read_csv(source_manifest)
    assert len(frame) == expected_rows, f"{split} rows {len(frame)} != {expected_rows}"
    counts = {name: int((frame["dx"] == name).sum()) for name in expected_counts}
    assert counts == expected_counts, f"{split} counts {counts} != {expected_counts}"
    assert "synthetic" not in set(frame.get("source", pd.Series(dtype=str)).astype(str)), f"{split} must be real-only"
    split_frames[split] = frame
    shutil.copy2(source_manifest, LOCAL_DATA_DIR / "manifests" / f"{split}.csv")
    copy_group(split, list(frame["image_path"].drop_duplicates()), SHARED_PROJECT_DIR / "data", LOCAL_DATA_DIR)
for field in ("image_id", "lesion_id"):
    assert not set(split_frames["train"][field]) & set(split_frames["val"][field]), f"train/val share {field}"
os.environ["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
if not DIAGNOSTIC_ROOT.exists():
    coca_run.ensure_tree(SHARED_RUN_ROOT, DIAGNOSTIC_ROOT.relative_to(SHARED_RUN_ROOT))
attempt_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = coca_run.ensure_tree(SHARED_RUN_ROOT, (DIAGNOSTIC_ROOT / attempt_id).relative_to(SHARED_RUN_ROOT))
assert str(OUTPUT_DIR).startswith(str(SHARED_RUN_ROOT)) and str(LOCAL_DATA_DIR) not in str(OUTPUT_DIR)
print("runtime-local data:", LOCAL_DATA_DIR)
print("drive attempt output:", OUTPUT_DIR)

## Phase 3: Encode images and run four analyses


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_DIR / "src")
env["PYTHONUNBUFFERED"] = "1"
command = [sys.executable, "-u", "-m", "ddpm_derm.coca_all_class_separability_diagnostic", "--output-dir", str(OUTPUT_DIR), "--git-commit", commit, "--shared-root-uuid", shared_root_uuid, "--device", "cuda", "--batch-size", "32", "--num-workers", "2"]
process = subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = []
for line in process.stdout:
    lines.append(line)
    print(line, end="", flush=True)
if process.wait():
    raise subprocess.CalledProcessError(process.returncode, command, output="".join(lines))

## Phase 4: Reopen and verify artifacts


In [ ]:
import numpy as np
from ddpm_derm import coca_all_class_separability_diagnostic as diag
record_path = OUTPUT_DIR / "all_class_separability_diagnostic.json"
completed_path = OUTPUT_DIR / "_COMPLETED.json"
identity_path = OUTPUT_DIR / "diagnostic_identity.json"
integrity_path = OUTPUT_DIR / "record_integrity.json"
embedding_path = OUTPUT_DIR / "all_class_embeddings.npz"
for path in (record_path, completed_path, identity_path, integrity_path, embedding_path, LATEST_RECORD):
    assert path.is_file(), f"missing artifact: {path}"
record_bytes = record_path.read_bytes()
assert record_bytes == completed_path.read_bytes() == LATEST_RECORD.read_bytes(), "formal records diverged"
record = json.loads(record_bytes)
assert record["diagnostic_status"] == "COMPLETED"
assert record["diagnostic_version"] == DIAGNOSTIC_VERSION
assert record["formal_training_started"] is False
assert record["test_data_accessed"] is False
assert record["condition_selected"] is False
assert record["interpretation_scope"] == "descriptive_representation_diagnostic_not_model_or_candidate_selection"
assert record["row_counts"] == {"train": EXPECTED_TRAIN_ROWS, "validation": EXPECTED_VALIDATION_ROWS}
assert record["class_counts"]["train"] == EXPECTED_TRAIN_CLASS_COUNTS
assert record["class_counts"]["validation"] == EXPECTED_VALIDATION_CLASS_COUNTS
assert "test" not in record["row_counts"] and "test" not in record["class_counts"]
assert record["feature_dimension"] == 512 and record["fixed_knn_k_values"] == [1, 5, 10]
assert set(record["analyses"]) == {"nearest_centroid", "cosine_knn", "validation_df_margin", "logistic_regression_probe"}
assert set(record["analyses"]["cosine_knn"]["by_k"]) == {"k1", "k5", "k10"}
assert record["analyses"]["logistic_regression_probe"]["converged"] is True

# --- record-integrity sidecar: recompute SHA-256, bytes, raw equality --------
integrity = diag.validate_record_integrity(output_dir=OUTPUT_DIR, latest_record_path=LATEST_RECORD)
assert integrity["schema"] == "all_class_separability_record_integrity_v1" and integrity["algorithm"] == "sha256"
formal_records = {"diagnostic_record": record_path, "completed_record": completed_path, "latest_record": LATEST_RECORD}
recomputed_hashes = set()
for name, path in formal_records.items():
    data = path.read_bytes()
    assert data == record_bytes, f"{name} raw bytes diverged"
    assert integrity[name]["sha256"] == sha256(path) == hashlib.sha256(data).hexdigest()
    assert integrity[name]["bytes"] == path.stat().st_size == len(data)
    recomputed_hashes.add(sha256(path))
assert len(recomputed_hashes) == 1 and integrity["raw_bytes_equal"] is True

# --- immutable diagnostic identity: rebuild the FULL expected identity --------
# Recompute the model identity from a freshly built frozen CoCa model (loaded on
# CPU from the warm HF cache), then rebuild the entire expected identity via the
# reviewed build_identity so validation compares model_identity too. This rejects
# a record/identity/top-level that are consistently wrong (e.g. a tampered eval
# preprocessing), not only the runtime-derived scalar fields.
identity = json.loads(identity_path.read_text(encoding="utf-8"))
assert record["diagnostic_identity"] == identity, "record diagnostic_identity != identity file"
from ddpm_derm.model import build_model, model_identity as compute_model_identity
review_model = build_model(arch="coca_vit_b32", freeze_backbone=True, coca_pretrained="laion2b_s13b_b90k")
expected_model_identity = compute_model_identity(review_model, "coca_vit_b32", 224)
del review_model
expected_identity = diag.build_identity(
    git_commit=commit,
    train_manifest_sha256=sha256(LOCAL_DATA_DIR / "manifests" / "train.csv"),
    validation_manifest_sha256=sha256(LOCAL_DATA_DIR / "manifests" / "val.csv"),
    shared_root_uuid=shared_root_uuid,
    diagnostic_output_identity=shared_root_uuid + ":sqrt_balanced_seed0_v1:coca_classifier:representation_diagnostics:" + DIAGNOSTIC_VERSION + ":" + OUTPUT_DIR.name,
    model_identity=expected_model_identity,
    dependency_versions={"open_clip_torch": version("open_clip_torch"), "torch": torch.__version__, "numpy": np.__version__, "scikit_learn": version("scikit-learn")},
    algorithm_identities=dict(diag.ALGORITHM_IDENTITIES),
)
diag.validate_diagnostic_identity(record, identity, expected_identity)
model_identity = identity["model_identity"]
assert model_identity == expected_model_identity, "recorded model_identity != freshly recomputed model identity"
assert record["model_identity"] == expected_model_identity, "record top-level model_identity != recomputed identity"
assert model_identity["arch"] == "coca_vit_b32"
assert model_identity["model_name"] == "coca_ViT-B-32"
assert model_identity["pretrained_tag"] == "laion2b_s13b_b90k"
assert model_identity["freeze_mode"] == "frozen_image_encoder_linear_head"
assert model_identity["input_resolution"] == [224, 224]
assert model_identity["preprocessing_identity"]["eval"] == expected_model_identity["preprocessing_identity"]["eval"]
assert identity["embedding_dimension"] == 512 == record["feature_dimension"]

# --- NPZ: independent eight-key dtype/shape/value + manifest-order contract ---
assert sha256(embedding_path) == record["embedding_artifact"]["sha256"]
assert embedding_path.stat().st_size == record["embedding_artifact"]["size_bytes"]
expected_npz_shapes = {"train_embeddings": (EXPECTED_TRAIN_ROWS, 512), "validation_embeddings": (EXPECTED_VALIDATION_ROWS, 512), "train_labels": (EXPECTED_TRAIN_ROWS,), "validation_labels": (EXPECTED_VALIDATION_ROWS,), "train_image_ids": (EXPECTED_TRAIN_ROWS,), "train_lesion_ids": (EXPECTED_TRAIN_ROWS,), "validation_image_ids": (EXPECTED_VALIDATION_ROWS,), "validation_lesion_ids": (EXPECTED_VALIDATION_ROWS,)}
manifest_frames = {"train": split_frames["train"], "validation": split_frames["val"]}
with np.load(embedding_path, allow_pickle=False) as saved:
    assert set(saved.files) == set(expected_npz_shapes), "NPZ must hold exactly the eight keys"
    for name, shape in expected_npz_shapes.items():
        assert tuple(saved[name].shape) == shape, f"{name} shape {saved[name].shape} != {shape}"
        assert saved[name].dtype != object, f"{name} must not be object dtype"
    for name in ("train_embeddings", "validation_embeddings"):
        assert saved[name].dtype == np.float32
        assert np.isfinite(saved[name]).all()
        np.testing.assert_allclose(np.linalg.norm(saved[name], axis=1), 1.0, rtol=1e-4, atol=1e-5)
    for split, prefix in (("train", "train"), ("validation", "validation")):
        frame = manifest_frames[split]
        assert saved[prefix + "_labels"].dtype == np.int64
        assert np.array_equal(saved[prefix + "_labels"], frame["label_idx"].to_numpy(dtype=np.int64))
        for field, key_suffix in (("image_id", "image_ids"), ("lesion_id", "lesion_ids")):
            key = prefix + "_" + key_suffix
            assert saved[key].dtype.kind == "U", f"{key} must be a unicode array"
            assert np.array_equal(saved[key], np.asarray(frame[field].astype(str).tolist(), dtype=np.str_))
    npz_keys = sorted(saved.files)
assert npz_keys == record["embedding_artifact"]["npz_keys"]
diag._validate_npz_file(embedding_path, n_train=EXPECTED_TRAIN_ROWS, n_val=EXPECTED_VALIDATION_ROWS)

# --- no checkpoints / formal artifacts; prior evidence still guarded ----------
checkpoints = [str(path) for path in OUTPUT_DIR.rglob("*.pt")]
assert not checkpoints, f"a representation diagnostic must not write model checkpoints: {checkpoints}"
guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
assert guard_after == guard_before, "prior v4/embedding/mixture evidence changed"
print("post-run review passed: identity, record-integrity sidecar, and eight-key NPZ all verified; prior evidence unchanged")

## Phase 5: Descriptive decision branches


In [ ]:
centroid_df_f1 = record["analyses"]["nearest_centroid"]["classification_summary"]["target_f1"]
knn_df_f1 = {key: record["analyses"]["cosine_knn"]["by_k"][key]["classification_summary"]["target_f1"] for key in ("k1", "k5", "k10")}
logistic_df_f1 = record["analyses"]["logistic_regression_probe"]["classification_summary"]["target_f1"]
margin = record["analyses"]["validation_df_margin"]
print("Nearest-centroid validation df F1:", centroid_df_f1)
print("Cosine k-NN validation df F1 by k:", knn_df_f1)
print("Fixed logistic-probe validation df F1:", logistic_df_f1)
print("Validation-df margin positive fraction:", margin["positive_margin_fraction"], "of", margin["validation_df_count"])
print()
print("Interpretation branches (descriptive only; this cell authorizes no next experiment):")
print("- If nearest-centroid, all k-NN, and the fixed logistic probe show no real validation df signal, a representation limitation is the priority to investigate next.")
print("- If the fixed logistic or non-parametric methods do recover df signal while the trained image-level head still collapses, the optimization / augmentation / objective is the priority.")
print("- Prior synthetic domain-gap evidence is described separately and must not be generalized into a claim that synthetic data is universally ineffective for other backbones.")
print()
print("ALL-CLASS COCA SEPARABILITY DIAGNOSTIC COMPLETED")
print("formal_training_started=false")
print("test_data_accessed=false")
print("condition_selected=false")
print("Download this executed notebook plus all_class_separability_diagnostic.json and all_class_embeddings.npz to archive with the record.")